# 05_evaluate

**Owner:** Models and evaluation. **Blueprint:** section 04, Evaluation. **Status:** Runs locally. Not yet run in Fabric.

Tests each model on the later deal cohorts. Their borrowers are not in training. Thresholds come from the release.

Reports accuracy, precision, recall, calibration and lift at ten reviews per run, and compares each model with the rules baseline. Lift compares the refinancing rate in the ten highest-scored accounts with the cohort base rate.

**Synthetic data.** HomeBridge is fictional. These results show that the workflow works. They do not show real predictive power or improved retention.

Output: `reports/evaluation_report.md` and `.json` under the output root.

In [ ]:
OUTPUT_ROOT = None  # default: data/output
MODEL_VERSION = None  # default: latest release

In [ ]:
import sys
from pathlib import Path

# Local run: this notebook sits in notebooks/ and imports notebooks/models_eval.
# Fabric run (not yet done): upload models_eval as a workspace resource or environment library,
# set HB_OUTPUT_ROOT to the Lakehouse Files path and HB_MLFLOW_URI = 'fabric'.
NOTEBOOKS_DIR = next(p for p in [Path.cwd(), Path.cwd() / 'notebooks'] if (p / 'models_eval').exists())
sys.path.insert(0, str(NOTEBOOKS_DIR))
import json
import pandas as pd
from models_eval import config
PATHS = config.paths(OUTPUT_ROOT)
print('Output root:', PATHS.root)

In [ ]:
from models_eval import evaluate
report = evaluate.evaluate_release(PATHS, MODEL_VERSION)
files = evaluate.write_report(PATHS, report)
files

## Model comparison on the test cohorts

In [ ]:
cols = ['role', 'threshold', 'accuracy', 'majority_class_accuracy', 'precision', 'recall', 'roc_auc', 'average_precision', 'mean_lift_at_10']
pd.DataFrame(report['models']).T[cols]

## Lift at ten reviews per run

One checkpoint run scores one deal-end cohort. Ten accounts is a small sample, so the bootstrap interval is wide.

In [ ]:
pd.DataFrame([{'model': m, **c} for m, d in report['models'].items() for c in d['lift_at_10_by_cohort']])

## Calibration

In [ ]:
pd.DataFrame([{'model': m, **b} for m, d in report['models'].items() for b in d['calibration']['bins']]).pivot(index='bin', columns='model', values=['mean_score', 'observed_rate']).round(3)

## Segments on the test cohorts

In [ ]:
pd.DataFrame(report['segments']['test_profiles'])

## Proxy check

Correlation of model inputs with context that the model does not use.

In [ ]:
pd.DataFrame(report['proxy_check']['results'])

## Full report

In [ ]:
from IPython.display import Markdown
Markdown(evaluate.to_markdown(report))